<font color="#CA0032"><h1 align="left">**Entradas heterogéneas**</h1></font>

<font color="#6E6E6E"><h1 align="left">**Predicción de ventas Rossmann**</h1></font>

<h2 align="left">Práctica B3—T5</h2>

**Componentes del grupo:**

- Nombre 1
- Nombre 2
- Nombre 3

### **Objetivo**

Construir un modelo neuronal para predicción de ventas usando entradas heterogéneas: series temporales de ventas, variables exógenas conocidas y atributos estáticos de cada tienda.

In [ ]:
COLAB = False

## <font color="#CA3532"> **1. Importar librerías**

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from IPython.display import display

In [ ]:
# Fijo la semilla aleatoria por reproducibilidad.
np.random.seed(7)

pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", "{:.3f}".format)

## <font color="#CA3532"> **2. Carga y unión de datos**

Los datos de Rossmann se distribuyen en tres ficheros principales:

- `train.csv`: histórico diario de ventas por tienda.
- `test.csv`: registros para los que se generará predicción.
- `store.csv`: información estática de cada tienda.

In [ ]:
DATASET_DIR = Path("dataset_completo_Rossmann-20260604T174330Z-3-001") / "dataset_completo_Rossmann"

if not DATASET_DIR.exists():
    candidates = sorted(Path(".").glob("**/dataset_completo_Rossmann"))
    if not candidates:
        raise FileNotFoundError("No se ha encontrado la carpeta dataset_completo_Rossmann")
    DATASET_DIR = candidates[0]

DATASET_DIR

In [ ]:
train_path = DATASET_DIR / "train.csv"
test_path = DATASET_DIR / "test.csv"
store_path = DATASET_DIR / "store.csv"
submission_path = DATASET_DIR / "submission.csv"

for path in [train_path, test_path, store_path, submission_path]:
    if not path.exists():
        raise FileNotFoundError(f"No se ha encontrado el fichero: {path}")

In [ ]:
date_columns = ["Date"]

train = pd.read_csv(train_path, parse_dates=date_columns, low_memory=False)
test = pd.read_csv(test_path, parse_dates=date_columns, low_memory=False)
store = pd.read_csv(store_path, low_memory=False)
submission = pd.read_csv(submission_path)

print("train:", train.shape)
print("test:", test.shape)
print("store:", store.shape)
print("submission:", submission.shape)

In [ ]:
display(train.head())
display(test.head())
display(store.head())

### **Revisión básica de estructura**

In [ ]:
def resumen_dataframe(df, nombre):
    return pd.Series({
        "filas": len(df),
        "columnas": df.shape[1],
        "tiendas": df["Store"].nunique() if "Store" in df else np.nan,
        "fecha_min": df["Date"].min() if "Date" in df else pd.NaT,
        "fecha_max": df["Date"].max() if "Date" in df else pd.NaT,
        "duplicados": df.duplicated().sum(),
    }, name=nombre)

resumen = pd.concat([
    resumen_dataframe(train, "train"),
    resumen_dataframe(test, "test"),
    resumen_dataframe(store, "store"),
], axis=1).T

resumen

In [ ]:
nulos = pd.concat({
    "train": train.isna().sum(),
    "test": test.isna().sum(),
    "store": store.isna().sum(),
}, axis=1).fillna(0).astype(int)

nulos[nulos.sum(axis=1) > 0]

### **Normalización inicial de tipos**

Antes de unir las tablas, se normalizan algunas variables categóricas para evitar inconsistencias entre `train`, `test` y `store`.

In [ ]:
def normalizar_state_holiday(df):
    df = df.copy()
    df["StateHoliday"] = df["StateHoliday"].astype(str).replace({"0.0": "0"})
    return df


def ordenar_por_tienda_y_fecha(df):
    return df.sort_values(["Store", "Date"]).reset_index(drop=True)


train = normalizar_state_holiday(train)
test = normalizar_state_holiday(test)

train = ordenar_por_tienda_y_fecha(train)
test = ordenar_por_tienda_y_fecha(test)
store = store.sort_values("Store").reset_index(drop=True)

### **Unión con atributos de tienda**

In [ ]:
if store["Store"].duplicated().any():
    raise ValueError("store.csv contiene tiendas duplicadas")

train_full = train.merge(store, on="Store", how="left", validate="many_to_one")
test_full = test.merge(store, on="Store", how="left", validate="many_to_one")

print("train_full:", train_full.shape)
print("test_full:", test_full.shape)

In [ ]:
columnas_sin_info_tienda = [
    col for col in store.columns
    if col != "Store" and train_full[col].isna().all()
]

if columnas_sin_info_tienda:
    raise ValueError(f"Columnas de tienda sin información tras el merge: {columnas_sin_info_tienda}")

display(train_full.head())
display(test_full.head())

### **Variables de calendario**

Se añaden variables derivadas de la fecha que podrán usarse como entradas exógenas conocidas antes del día de predicción.

In [ ]:
def crear_variables_calendario(df):
    df = df.copy()
    df["Year"] = df["Date"].dt.year
    df["Month"] = df["Date"].dt.month
    df["Day"] = df["Date"].dt.day
    df["WeekOfYear"] = df["Date"].dt.isocalendar().week.astype(int)
    df["DayOfYear"] = df["Date"].dt.dayofyear
    return df


train_full = crear_variables_calendario(train_full)
test_full = crear_variables_calendario(test_full)

In [ ]:
columnas_base = [
    "Store", "Date", "Sales", "Customers", "Open", "Promo",
    "StateHoliday", "SchoolHoliday", "StoreType", "Assortment",
    "CompetitionDistance", "Promo2", "Year", "Month", "DayOfWeek",
    "WeekOfYear", "DayOfYear",
]

display(train_full[columnas_base].head(10))

### **Resumen para el modelado**

In [ ]:
variables_endogenas = ["Sales"]
variables_exogenas_temporales = [
    "Customers", "Open", "Promo", "StateHoliday", "SchoolHoliday",
    "DayOfWeek", "Month", "WeekOfYear", "DayOfYear",
]
variables_estaticas_tienda = [
    "Store", "StoreType", "Assortment", "CompetitionDistance",
    "CompetitionOpenSinceMonth", "CompetitionOpenSinceYear",
    "Promo2", "Promo2SinceWeek", "Promo2SinceYear", "PromoInterval",
]

pd.DataFrame({
    "tipo": [
        "endógenas",
        "exógenas temporales",
        "estáticas de tienda",
    ],
    "variables": [
        variables_endogenas,
        variables_exogenas_temporales,
        variables_estaticas_tienda,
    ],
})

## <font color="#CA3532"> **3. Limpieza avanzada y variables temporales**

En este bloque se preparan variables útiles para el modelado: tratamiento inicial de nulos, variables de competencia, variables de promoción continuada y codificación básica de categorías. El objetivo es dejar una tabla limpia antes del enventanado temporal.

In [ ]:
def limpiar_variables_tienda(df, distancia_mediana=None):
    df = df.copy()

    if distancia_mediana is None:
        distancia_mediana = df["CompetitionDistance"].median()

    df["CompetitionDistance"] = df["CompetitionDistance"].fillna(distancia_mediana)
    df["PromoInterval"] = df["PromoInterval"].fillna("None")

    columnas_categoricas = ["StateHoliday", "StoreType", "Assortment", "PromoInterval"]
    for columna in columnas_categoricas:
        df[columna] = df[columna].fillna("Unknown").astype(str)

    columnas_numericas_con_nulos = [
        "CompetitionOpenSinceMonth", "CompetitionOpenSinceYear",
        "Promo2SinceWeek", "Promo2SinceYear",
    ]
    for columna in columnas_numericas_con_nulos:
        df[columna] = df[columna].fillna(0).astype(int)

    return df, distancia_mediana


train_full, distancia_mediana = limpiar_variables_tienda(train_full)
test_full, _ = limpiar_variables_tienda(test_full, distancia_mediana=distancia_mediana)

distancia_mediana

### **Variables de competencia**

`CompetitionOpenSinceMonth` y `CompetitionOpenSinceYear` indican desde cuándo hay competencia cercana. Se transforman en una variable temporal más directa: meses desde que abrió la competencia.

In [ ]:
def crear_variables_competencia(df):
    df = df.copy()

    hay_fecha_competencia = (
        (df["CompetitionOpenSinceMonth"] > 0)
        & (df["CompetitionOpenSinceYear"] > 0)
    )

    meses_desde_competencia = (
        (df["Year"] - df["CompetitionOpenSinceYear"]) * 12
        + (df["Month"] - df["CompetitionOpenSinceMonth"])
    )

    df["CompetitionOpen"] = hay_fecha_competencia.astype(int)
    df["CompetitionOpenMonths"] = meses_desde_competencia.where(hay_fecha_competencia, 0)
    df["CompetitionOpenMonths"] = df["CompetitionOpenMonths"].clip(lower=0)

    return df


train_full = crear_variables_competencia(train_full)
test_full = crear_variables_competencia(test_full)

train_full[[
    "Store", "Date", "CompetitionDistance", "CompetitionOpen",
    "CompetitionOpenSinceMonth", "CompetitionOpenSinceYear",
    "CompetitionOpenMonths",
]].head(10)

### **Variables de promoción continuada**

`Promo2` indica si una tienda participa en una promoción continuada. A partir de sus columnas asociadas se crean dos variables: semanas desde el inicio de `Promo2` y si la promoción está activa en el mes del registro.

In [ ]:
MESES_PROMO = {
    1: "Jan", 2: "Feb", 3: "Mar", 4: "Apr", 5: "May", 6: "Jun",
    7: "Jul", 8: "Aug", 9: "Sept", 10: "Oct", 11: "Nov", 12: "Dec",
}


def crear_variables_promo2(df):
    df = df.copy()

    hay_inicio_promo2 = (
        (df["Promo2"] == 1)
        & (df["Promo2SinceWeek"] > 0)
        & (df["Promo2SinceYear"] > 0)
    )

    inicio_promo2 = pd.to_datetime(
        df["Promo2SinceYear"].astype(str)
        + df["Promo2SinceWeek"].astype(str).str.zfill(2)
        + "1",
        format="%G%V%u",
        errors="coerce",
    )

    semanas_promo2 = ((df["Date"] - inicio_promo2).dt.days // 7).where(hay_inicio_promo2, 0)
    df["Promo2Weeks"] = semanas_promo2.fillna(0).clip(lower=0).astype(int)

    mes_actual = df["Month"].map(MESES_PROMO)
    df["Promo2ActiveMonth"] = [
        int(promo2 == 1 and mes in intervalo.split(","))
        for promo2, mes, intervalo in zip(df["Promo2"], mes_actual, df["PromoInterval"])
    ]

    return df


train_full = crear_variables_promo2(train_full)
test_full = crear_variables_promo2(test_full)

train_full[[
    "Store", "Date", "Promo2", "Promo2SinceWeek", "Promo2SinceYear",
    "PromoInterval", "Promo2Weeks", "Promo2ActiveMonth",
]].head(10)

### **Códigos categóricos para embeddings**

Las variables categóricas se convierten a índices enteros. Estos índices podrán usarse después como entrada de capas `Embedding`.

In [ ]:
def crear_codigos_categoricos(train_df, test_df, columnas):
    train_df = train_df.copy()
    test_df = test_df.copy()
    cardinalidades = {}

    for columna in columnas:
        categorias = pd.Index(
            pd.concat([train_df[columna], test_df[columna]], ignore_index=True)
            .astype(str)
            .unique()
        ).sort_values()

        mapa = {categoria: codigo for codigo, categoria in enumerate(categorias)}
        columna_codigo = f"{columna}Code"

        train_df[columna_codigo] = train_df[columna].astype(str).map(mapa).astype(int)
        test_df[columna_codigo] = test_df[columna].astype(str).map(mapa).astype(int)
        cardinalidades[columna_codigo] = len(categorias)

    return train_df, test_df, cardinalidades


columnas_categoricas_embedding = [
    "Store", "DayOfWeek", "StateHoliday", "StoreType", "Assortment", "PromoInterval",
]

train_full, test_full, cardinalidades_embedding = crear_codigos_categoricos(
    train_full,
    test_full,
    columnas_categoricas_embedding,
)

cardinalidades_embedding

## <font color="#CA3532"> **4. Enventanado temporal por tienda**

El enventanado se hace tienda a tienda para no mezclar el final de una serie temporal con el inicio de otra. Cada muestra contiene los últimos `lookback` días de información y el objetivo es `Sales` en el día siguiente.

In [ ]:
columnas_temporales_ventana = [
    "Sales", "Customers", "Open", "Promo", "SchoolHoliday",
    "CompetitionDistance", "CompetitionOpenMonths",
    "Promo2Weeks", "Promo2ActiveMonth",
    "DayOfWeek", "Month", "WeekOfYear", "DayOfYear",
]

columnas_estaticas_ventana = [
    "StoreCode", "StoreTypeCode", "AssortmentCode", "PromoIntervalCode",
    "CompetitionDistance", "CompetitionOpen", "Promo2",
]

target = "Sales"
lookback = 28
horizonte = 1

# Para desarrollar y validar el flujo usamos 10 tiendas.
# Cambiar a None para generar ventanas de todas las tiendas.
tiendas_desarrollo = sorted(train_full["Store"].unique())[:10]

In [ ]:
if tiendas_desarrollo is None:
    train_modelado = train_full.copy()
else:
    train_modelado = train_full[train_full["Store"].isin(tiendas_desarrollo)].copy()

print("tiendas usadas:", train_modelado["Store"].nunique())
print("filas usadas:", len(train_modelado))

In [ ]:
def crear_ventanas_por_tienda(
    df,
    columnas_temporales,
    columna_target="Sales",
    columnas_estaticas=None,
    lookback=28,
    horizonte=1,
):
    if columnas_estaticas is None:
        columnas_estaticas = []

    X_temporal = []
    X_estatico = []
    y = []
    metadatos = []

    columnas_necesarias = list(dict.fromkeys(
        ["Store", "Date", columna_target] + columnas_temporales + columnas_estaticas
    ))
    df_ordenado = df[columnas_necesarias].sort_values(["Store", "Date"]).reset_index(drop=True)

    for store_id, datos_tienda in df_ordenado.groupby("Store", sort=False):
        datos_tienda = datos_tienda.sort_values("Date").reset_index(drop=True)
        valores_temporales = datos_tienda[columnas_temporales].to_numpy(dtype="float32")
        valores_target = datos_tienda[columna_target].to_numpy(dtype="float32")
        valores_estaticos = datos_tienda[columnas_estaticas].to_numpy(dtype="float32") if columnas_estaticas else None

        primer_indice_objetivo = lookback + horizonte - 1
        for indice_objetivo in range(primer_indice_objetivo, len(datos_tienda)):
            fin_ventana = indice_objetivo - horizonte + 1
            inicio_ventana = fin_ventana - lookback

            X_temporal.append(valores_temporales[inicio_ventana:fin_ventana])
            y.append(valores_target[indice_objetivo])

            if columnas_estaticas:
                X_estatico.append(valores_estaticos[indice_objetivo])

            metadatos.append({
                "Store": store_id,
                "Date": datos_tienda.loc[indice_objetivo, "Date"],
            })

    X_temporal = np.asarray(X_temporal, dtype="float32")
    y = np.asarray(y, dtype="float32")
    metadatos = pd.DataFrame(metadatos)

    if columnas_estaticas:
        X_estatico = np.asarray(X_estatico, dtype="float32")
    else:
        X_estatico = None

    return X_temporal, X_estatico, y, metadatos


X_temporal, X_estatico, y, metadatos_ventanas = crear_ventanas_por_tienda(
    train_modelado,
    columnas_temporales=columnas_temporales_ventana,
    columna_target=target,
    columnas_estaticas=columnas_estaticas_ventana,
    lookback=lookback,
    horizonte=horizonte,
)

print("X_temporal:", X_temporal.shape)
print("X_estatico:", X_estatico.shape)
print("y:", y.shape)
display(metadatos_ventanas.head())

### **Separación temporal de entrenamiento y validación**

La validación se separa por fecha, respetando el orden temporal. Así evitamos entrenar con datos posteriores a los que queremos validar.

In [ ]:
fecha_corte_validacion = train_full["Date"].max() - pd.Timedelta(days=42)

mascara_train = metadatos_ventanas["Date"] <= fecha_corte_validacion
mascara_val = metadatos_ventanas["Date"] > fecha_corte_validacion

X_temporal_train = X_temporal[mascara_train]
X_temporal_val = X_temporal[mascara_val]
X_estatico_train = X_estatico[mascara_train]
X_estatico_val = X_estatico[mascara_val]
y_train = y[mascara_train]
y_val = y[mascara_val]

print("fecha_corte_validacion:", fecha_corte_validacion.date())
print("X_temporal_train:", X_temporal_train.shape)
print("X_temporal_val:", X_temporal_val.shape)
print("X_estatico_train:", X_estatico_train.shape)
print("X_estatico_val:", X_estatico_val.shape)
print("y_train:", y_train.shape)
print("y_val:", y_val.shape)

Con este bloque quedan preparadas las principales entradas del futuro modelo: `X_temporal` para la rama recurrente, `X_estatico` para atributos de tienda y `y` como objetivo de ventas. El siguiente paso será escalar las variables numéricas y construir la arquitectura neuronal con entradas heterogéneas.